# Inference + Evaluation — SEC Filing Extraction (Colab GPU)

Loads the trained QLoRA adapter and runs extraction against sample SEC filings.

**What this notebook does:**
1. Loads the fine-tuned model (base + LoRA adapter from Google Drive)
2. Extracts structured data from a sample 10-K filing
3. Runs the 5-strategy JSON parser on model output
4. Evaluates per-field accuracy against ground truth
5. Profiles inference latency and GPU memory usage

## 1. Environment Setup

In [ ]:
%%capture
!pip install torch transformers peft bitsandbytes accelerate
!pip install sentencepiece protobuf loguru pyyaml python-dotenv tqdm rich

import os
REPO_URL = "https://github.com/akuo6/financial-llm.git"
REPO_DIR = "/content/FinDocAnalyzer"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

import sys
sys.path.insert(0, REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 2. GPU Check + HuggingFace Auth

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Go to Runtime → Change runtime type → GPU.")

gpu = torch.cuda.get_device_properties(0)
vram_gb = gpu.total_mem / 1e9
print(f"GPU: {gpu.name} ({vram_gb:.1f} GB VRAM)")

from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    hf_token = input("Enter your HuggingFace token: ")

login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
print("Authenticated.")

## 3. Load Model + Adapter

Loads the NF4-quantized base model and merges the LoRA adapter.

Set `ADAPTER_SOURCE` below:
- `"drive"` — load from Google Drive (saved by `train_qlora.ipynb`)
- `"local"` — load from the repo's default path (if adapter was just trained in this session)

In [ ]:
ADAPTER_SOURCE = "drive"  # "drive" or "local"

if ADAPTER_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    adapter_path = "/content/drive/MyDrive/FinDocAnalyzer/models/llama-sec-v1"
else:
    adapter_path = os.path.join(REPO_DIR, "models/llama-sec-v1")

from pathlib import Path
adapter_dir = Path(adapter_path)
if adapter_dir.exists() and (adapter_dir / "adapter_config.json").exists():
    print(f"Adapter found at: {adapter_path}")
    adapter_files = list(adapter_dir.glob("*"))
    total_mb = sum(f.stat().st_size for f in adapter_files if f.is_file()) / 1e6
    print(f"Adapter size: {total_mb:.1f} MB ({len(adapter_files)} files)")
else:
    print(f"WARNING: No adapter at {adapter_path}")
    print("The model will run with base weights only (lower accuracy).")

In [ ]:
from src.model import FinancialLLM
import time

print("Loading model (NF4 quantized + LoRA adapter)...")
load_start = time.time()

llm = FinancialLLM.from_pretrained(
    adapter_path=adapter_path,
    merge_adapter=True,
)

load_time = time.time() - load_start
mem_gb = torch.cuda.memory_allocated() / 1e9
print(f"\nModel loaded in {load_time:.1f}s")
print(f"GPU memory: {mem_gb:.2f} GB / {vram_gb:.1f} GB ({100 * mem_gb / vram_gb:.0f}%)")

## 4. Run Extraction on Sample 10-K

Uses the ExtractionEngine to process the sample SEC filing and display the structured output.

In [ ]:
from src.inference import ExtractionEngine, ExtractionRequest
import json

engine = ExtractionEngine(model=llm)

sample_path = os.path.join(REPO_DIR, "data", "sample_10k.txt")
with open(sample_path) as f:
    sample_text = f.read()

print(f"Sample filing: {len(sample_text)} chars")
print(f"First 200 chars:\n{sample_text[:200]}...")
print("\n" + "=" * 60)

# Run extraction
request = ExtractionRequest(text=sample_text)
response = engine.extract(request)

print(f"\nStatus:     {response.status}")
print(f"Latency:    {response.latency_ms:.0f} ms")
print(f"Confidence: {response.confidence_score:.2f}")
print(f"\nExtracted JSON:")
if response.result:
    print(json.dumps(response.result.__dict__, indent=2, default=str))
else:
    print(f"Error: {response.error}")
    print(f"Raw output:\n{response.raw_output[:500]}")

## 5. Evaluate Against Ground Truth

Compares extracted fields against the expected output for the sample filing.

In [ ]:
expected_path = os.path.join(REPO_DIR, "data", "sample_10k.expected.json")
with open(expected_path) as f:
    expected = json.load(f)

if response.result:
    predicted = response.result.__dict__
    fields = list(expected.keys())

    correct = 0
    total = 0
    print(f"{'Field':<22} {'Expected':<25} {'Predicted':<25} {'Match'}")
    print("-" * 90)

    for field in fields:
        exp_val = expected.get(field)
        pred_val = predicted.get(field)
        total += 1

        # Fuzzy numeric matching (±5% tolerance)
        if isinstance(exp_val, (int, float)) and isinstance(pred_val, (int, float)):
            if exp_val == 0:
                match = pred_val == 0
            else:
                match = abs(pred_val - exp_val) / abs(exp_val) <= 0.05
        else:
            match = str(exp_val).lower().strip() == str(pred_val).lower().strip()

        correct += int(match)
        symbol = "✓" if match else "✗"
        print(f"{field:<22} {str(exp_val):<25} {str(pred_val):<25} {symbol}")

    print("-" * 90)
    print(f"Field accuracy: {correct}/{total} ({100 * correct / total:.0f}%)")
else:
    print("No extraction result to evaluate.")

## 6. Latency Profiling

Runs the model multiple times to measure stable inference latency with GPU warm-up.

In [ ]:
import numpy as np

N_WARMUP = 3
N_RUNS = 10

print(f"Warm-up ({N_WARMUP} runs)...")
for _ in range(N_WARMUP):
    engine.extract(ExtractionRequest(text=sample_text[:2000]))

print(f"Benchmarking ({N_RUNS} runs)...")
latencies = []
for i in range(N_RUNS):
    resp = engine.extract(ExtractionRequest(text=sample_text))
    latencies.append(resp.latency_ms)
    print(f"  Run {i+1}: {resp.latency_ms:.0f} ms ({resp.status})")

latencies_np = np.array(latencies)
print(f"\n{'=' * 40}")
print(f"Mean:   {latencies_np.mean():.0f} ms")
print(f"Median: {np.median(latencies_np):.0f} ms")
print(f"P95:    {np.percentile(latencies_np, 95):.0f} ms")
print(f"P99:    {np.percentile(latencies_np, 99):.0f} ms")
print(f"Std:    {latencies_np.std():.0f} ms")
print(f"\nThroughput: ~{60_000 / latencies_np.mean():.0f} docs/min")

## 7. GPU Memory Profile

In [ ]:
stats = llm.get_memory_stats()
print("GPU Memory Usage:")
print(f"  Allocated:     {stats['allocated_mb']:.0f} MB")
print(f"  Reserved:      {stats['reserved_mb']:.0f} MB")
print(f"  Peak:          {stats['max_allocated_mb']:.0f} MB")
print(f"  Total VRAM:    {vram_gb * 1000:.0f} MB")
print(f"  Utilization:   {100 * stats['allocated_mb'] / (vram_gb * 1000):.0f}%")

# CUDA kernel time breakdown (requires torch profiler)
print(f"\nFor reference (NF4 quantized Llama 3.1 8B):")
print(f"  FP32 baseline: ~32,000 MB")
print(f"  Current NF4:   ~{stats['allocated_mb']:.0f} MB")
print(f"  Reduction:     {100 * (1 - stats['allocated_mb'] / 32000):.0f}%")

## 8. Batch Extraction (Test Set)

Runs extraction against the full synthetic test set to compute aggregate accuracy.

In [ ]:
test_path = os.path.join(REPO_DIR, "data", "sec_filings_test.jsonl")

if not os.path.exists(test_path):
    print("Test set not found. Generating...")
    !python scripts/download_dataset.py

if os.path.exists(test_path):
    test_docs = []
    with open(test_path) as f:
        for line in f:
            test_docs.append(json.loads(line))

    MAX_EVAL = 20
    docs_to_eval = test_docs[:MAX_EVAL]
    print(f"Evaluating on {len(docs_to_eval)} / {len(test_docs)} test documents...\n")

    from tqdm import tqdm
    results = []
    for doc in tqdm(docs_to_eval, desc="Extracting"):
        text = doc.get("input", doc.get("text", ""))
        req = ExtractionRequest(text=text)
        resp = engine.extract(req)
        results.append({
            "status": resp.status,
            "latency_ms": resp.latency_ms,
            "confidence": resp.confidence_score,
            "result": resp.result.__dict__ if resp.result else None,
        })

    n_success = sum(1 for r in results if r["status"] == "success")
    avg_latency = np.mean([r["latency_ms"] for r in results])
    avg_confidence = np.mean([r["confidence"] for r in results])

    print(f"\nResults ({len(docs_to_eval)} documents):")
    print(f"  Success rate:     {n_success}/{len(docs_to_eval)} ({100 * n_success / len(docs_to_eval):.0f}%)")
    print(f"  Avg latency:      {avg_latency:.0f} ms")
    print(f"  Avg confidence:   {avg_confidence:.2f}")
    print(f"  Est. throughput:  ~{60_000 / avg_latency:.0f} docs/min")
else:
    print("Could not generate test data. Skipping batch evaluation.")